    Esta celda solamente prepara las librerías, 
    fija la semilla para que los resultados sean reproducibles y selecciona GPU si está disponible.

In [1]:
## Learning targets

# 1. Make LR-ASPP emit two class-score channels at each pixel.
# 2. Freeze the visual backbone and optimize only the new classifier head.
# 3. Save a checkpoint plus validation predictions that Session 5 can inspect without retraining.

import sys
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision

from torchvision.datasets import OxfordIIITPet
from torchvision.models.segmentation import (
    LRASPP_MobileNet_V3_Large_Weights,
    lraspp_mobilenet_v3_large
)
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF


SEED = 17
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 8
EPOCHS = 1

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = ARTIFACT_DIR / "session4_baseline.pt"
PREPARED_BASELINE = ARTIFACT_DIR / "instructor_baseline.pt"

print(
    f"torch={torch.__version__} | "
    f"torchvision={torchvision.__version__} | "
    f"device={device} | seed={SEED}"
)

torch=2.12.1+cpu | torchvision=0.27.1+cpu | device=cpu | seed=17


    Aquí se descargan los datos de Oxford-IIIT Pet y se seleccionan exactamente:

    64 imágenes para entrenamiento.
    24 imágenes para validación.

    La semilla 17 hace que todos obtengan la misma división.

In [2]:
## 1. Recreate the fixed split

# Set OXFORD_PET_ROOT to the directory containing
# oxford-iiit-pet/ when data is pre-cached.

DATA_ROOT = Path(os.environ.get("OXFORD_PET_ROOT", "data"))

base = OxfordIIITPet(
    root=DATA_ROOT,
    split="trainval",
    target_types="segmentation",
    download=True
)

permutation = torch.randperm(
    len(base),
    generator=torch.Generator().manual_seed(SEED)
).tolist()

train_indices = permutation[:64]
val_indices = permutation[64:88]

assert len(train_indices) == 64
assert len(val_indices) == 24
assert not set(train_indices).intersection(val_indices)

print(
    f"fixed split: train={len(train_indices)}, "
    f"validation={len(val_indices)}, overlap=0"
)

print("first five train indices:", train_indices[:5])

fixed split: train=64, validation=24, overlap=0
first five train indices: [2479, 1638, 733, 859, 670]


    La máscara original de Oxford-IIIT Pet tiene tres valores. 
    Aquí se transforma a:

    0   = background
    1   = pet
    255 = boundary / ignored

    También las imágenes y máscaras se redimensionan a 128 × 128.

    "BILINEAR" para las imágenes y "NEAREST" para las máscaras. Usa interpolación bilineal en una máscara podría crear valores de clase que no existen.

In [3]:
def remap_trimap(raw_mask):
    raw = torch.as_tensor(
        np.asarray(raw_mask, dtype=np.uint8),
        dtype=torch.long
    )

    target = torch.full_like(raw, 255)

    target[raw == 1] = 1  # pet
    target[raw == 2] = 0  # background

    return target


def paired_transform(image, raw_mask):
    image = TF.resize(
        image,
        IMAGE_SIZE,
        interpolation=InterpolationMode.BILINEAR,
        antialias=True
    )

    raw_mask = TF.resize(
        raw_mask,
        IMAGE_SIZE,
        interpolation=InterpolationMode.NEAREST
    )

    image = TF.to_tensor(image)
    image = TF.normalize(image, mean=MEAN, std=STD)

    return image, remap_trimap(raw_mask)


class PetSubset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        image, trimap = self.dataset[self.indices[i]]
        return paired_transform(image, trimap)


train_loader = DataLoader(
    PetSubset(base, train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    PetSubset(base, val_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


images, masks = next(iter(train_loader))

assert images.shape == (BATCH_SIZE, 3, 128, 128)
assert masks.shape == (BATCH_SIZE, 128, 128)
assert set(torch.unique(masks).tolist()) <= {0, 1, 255}

print(
    "batch contract:",
    tuple(images.shape),
    tuple(masks.shape),
    torch.unique(masks).tolist()
)

batch contract: (8, 3, 128, 128) (8, 128, 128) [0, 1, 255]


C:\Users\tiori\AppData\Local\Temp\ipykernel_23244\3214017206.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  raw = torch.as_tensor(


    Aquí se cumple el Learning Target 1.

    El modelo original está preparado para varias clases, para 2:

    canal 0 → background
    canal 1 → pet

8   = imágenes
2   = clases
128 = alto
128 = ancho    

    No se utiliza softmax durante el entrenamiento porque F.cross_entropy() espera directamente los logits.

In [4]:
## 2. Replace the output head

weights = LRASPP_MobileNet_V3_Large_Weights.DEFAULT

model = lraspp_mobilenet_v3_large(weights=weights)

model.classifier.low_classifier = nn.Conv2d(
    model.classifier.low_classifier.in_channels,
    2,
    kernel_size=1
)

model.classifier.high_classifier = nn.Conv2d(
    model.classifier.high_classifier.in_channels,
    2,
    kernel_size=1
)

model = model.to(device)


with torch.no_grad():
    logits = model(images.to(device))["out"]


print("logits:", tuple(logits.shape))

assert logits.shape == (BATCH_SIZE, 2, 128, 128)


Downloading: "https://download.pytorch.org/models/lraspp_mobilenet_v3_large-d234d4ea.pth" to C:\Users\tiori/.cache\torch\hub\checkpoints\lraspp_mobilenet_v3_large-d234d4ea.pth


100.0%


logits: (8, 2, 128, 128)


# Congelar el backbone

    Esta celda cumple el Learning Target 2.

    El backbone de MobileNetV3 conserva el conocimiento aprendido durante su preentrenamiento.

    Solamente se entrenarán los parámetros del: "classifier"

    El optimizador tampoco recibe todos los parámetros del modelo, sino únicamente aquellos con:
    requires_grad == True

In [5]:
## 3. Freeze intentionally

for parameter in model.backbone.parameters():
    parameter.requires_grad = False


trainable = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]


optimizer = torch.optim.Adam(
    trainable,
    lr=1e-3
)


print("trainable names:", trainable_names)


assert trainable_names
assert all(
    name.startswith("classifier.")
    for name in trainable_names
)

assert all(
    not parameter.requires_grad
    for parameter in model.backbone.parameters()
)


trainable names: ['classifier.cbr.0.weight', 'classifier.cbr.1.weight', 'classifier.cbr.1.bias', 'classifier.scale.1.weight', 'classifier.low_classifier.weight', 'classifier.low_classifier.bias', 'classifier.high_classifier.weight', 'classifier.high_classifier.bias']


# Evaluación antes del entrenamiento

    Aqui se calcula:

    loss
    pixel_accuracy
    pet_iou

In [6]:
## 4. Measure a pre-training baseline

def evaluate(model, loader, keep_predictions=False):

    model.eval()

    total_loss = 0.0
    valid_pixels = 0
    correct = 0
    intersection = 0
    union = 0

    saved_images = []
    saved_targets = []
    saved_predictions = []
    saved_logits = []

    with torch.no_grad():

        for batch_images, targets in loader:

            batch_images = batch_images.to(device)
            targets = targets.to(device)

            logits = model(batch_images)["out"]

            total_loss += F.cross_entropy(
                logits,
                targets,
                ignore_index=255,
                reduction="sum"
            ).item()

            predictions = logits.argmax(dim=1)

            valid = targets != 255

            valid_pixels += valid.sum().item()

            correct += (
                (predictions == targets) & valid
            ).sum().item()

            intersection += (
                (predictions == 1) &
                (targets == 1) &
                valid
            ).sum().item()

            union += (
                ((predictions == 1) | (targets == 1)) &
                valid
            ).sum().item()

            if keep_predictions:
                saved_images.append(batch_images.cpu())
                saved_targets.append(targets.cpu())
                saved_predictions.append(predictions.cpu())
                saved_logits.append(logits.cpu())


    result = {
        "loss": total_loss / max(valid_pixels, 1),
        "pixel_accuracy": correct / max(valid_pixels, 1),
        "pet_iou": intersection / max(union, 1)
    }


    if keep_predictions:
        result.update(
            images=torch.cat(saved_images),
            targets=torch.cat(saved_targets),
            predictions=torch.cat(saved_predictions),
            logits=torch.cat(saved_logits)
        )

    return result


before = evaluate(model, val_loader)

print({
    key: round(value, 4)
    for key, value in before.items()
})

assert all(
    np.isfinite(value)
    for value in before.values()
)

{'loss': 0.6477, 'pixel_accuracy': 0.6711, 'pet_iou': 0.0764}


# Entrenar una época y guardar resultados

forward
   ↓
cross entropy
   ↓
backward
   ↓
optimizer.step()


    Como el backbone está congelado, los pesos que cambian pertenecen solamente al clasificador.

Después se guarda:

    artifacts/session4_baseline.pt

y dentro del archivo quedan tanto los pesos como información que la siguiente sesión podrá inspeccionar sin volver a entrenar.

En concreto, se almacenan los índices de entrenamiento y validación, métricas, pesos, imágenes de validación, máscaras reales, predicciones y logits.

In [7]:
## 5. Fine-tune one epoch and save the handoff artifact

model.train()

epoch_losses = []


for batch_images, targets in train_loader:

    batch_images = batch_images.to(device)
    targets = targets.to(device)

    optimizer.zero_grad()

    logits = model(batch_images)["out"]

    loss = F.cross_entropy(
        logits,
        targets,
        ignore_index=255
    )

    loss.backward()

    optimizer.step()

    epoch_losses.append(loss.item())


mean_train_loss = float(np.mean(epoch_losses))


after = evaluate(
    model,
    val_loader,
    keep_predictions=True
)


artifact = {
    "artifact_version": 1,
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "epochs": EPOCHS,

    "train_indices": train_indices,
    "val_indices": val_indices,

    "mean_train_loss": mean_train_loss,

    "metrics": {
        key: value
        for key, value in after.items()
        if isinstance(value, float)
    },

    "model_state_dict": {
        key: value.cpu()
        for key, value in model.state_dict().items()
    },

    "validation_images": after["images"],
    "validation_targets": after["targets"],
    "validation_predictions": after["predictions"],
    "validation_logits": after["logits"],
}


torch.save(
    artifact,
    BASELINE_PATH
)


print(
    f"one epoch mean loss={mean_train_loss:.4f}"
)

print(
    "saved Session 5 handoff:",
    BASELINE_PATH.resolve()
)

print({
    key: round(value, 4)
    for key, value in artifact["metrics"].items()
})


assert BASELINE_PATH.exists()
assert np.isfinite(mean_train_loss)

one epoch mean loss=0.5087
saved Session 5 handoff: C:\Module4_IA_Cinvestav\Specialization_Artificial_Intelligence_Cinvestav\MarioValdovinos_Module4\artifacts\session4_baseline.pt
{'loss': 0.3677, 'pixel_accuracy': 0.8504, 'pet_iou': 0.636}


# Comprobar el archivo generado

Esta última celda verifica que el archivo esté disponible. Primero intenta usar:

    artifacts/session4_baseline.pt

Si no existe, busca:

    artifacts/instructor_baseline.pt

In [8]:
handoff_path = (
    BASELINE_PATH
    if BASELINE_PATH.exists()
    else PREPARED_BASELINE
)


if handoff_path.exists():

    saved = torch.load(
        handoff_path,
        map_location="cpu",
        weights_only=False
    )

    print(
        f"Session 5 should load: "
        f"{handoff_path.resolve()}"
    )

    print(
        "saved seed/epochs:",
        saved["seed"],
        saved["epochs"]
    )

else:

    print(
        "No prepared fallback yet. "
        "Ask the instructor for "
        "artifacts/instructor_baseline.pt."
    )

Session 5 should load: C:\Module4_IA_Cinvestav\Specialization_Artificial_Intelligence_Cinvestav\MarioValdovinos_Module4\artifacts\session4_baseline.pt
saved seed/epochs: 17 1
